# Honeydew: 2,500 MIDI + 2,500 WAV training set (fixed)
This notebook creates 2,500 original short compositions as MIDI, renders each to WAV, builds a manifest, and trains a small next-event LSTM baseline. The corpus is algorithmically generated; it is useful for pipeline validation, not a substitute for licensed human-composed music.

In [ ]:
!apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm ffmpeg
!pip -q install pretty_midi soundfile torch tqdm pandas
import os, json, random, subprocess, math, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, pretty_midi, torch
from tqdm.auto import tqdm
random.seed(20261121); np.random.seed(20261121); torch.manual_seed(20261121)
ROOT=Path('/content/honeydew_dataset'); MIDI_DIR=ROOT/'midi'; WAV_DIR=ROOT/'wav'
MIDI_DIR.mkdir(parents=True, exist_ok=True); WAV_DIR.mkdir(parents=True, exist_ok=True)
candidates=[
    '/usr/share/sounds/sf2/FluidR3_GM.sf2',
    '/usr/share/sounds/sf2/FluidR3_GM.sf3',
    '/usr/share/soundfonts/default-GM.sf2'
]
found=[]
for base in ['/usr/share/sounds','/usr/share/soundfonts']:
    if os.path.isdir(base):
        for root, _, files in os.walk(base):
            found.extend(os.path.join(root, name) for name in files if name.lower().endswith(('.sf2','.sf3')))
SOUNDFONT=next((p for p in candidates+found if Path(p).exists()), None)
assert shutil.which('fluidsynth'), 'FluidSynth installation failed'
assert SOUNDFONT, 'No GM soundfont found; check fluid-soundfont-gm installation'
print('Using soundfont:', SOUNDFONT)

## Generate 2,500 original MIDI compositions and render matching WAV files
Each piece is 8 bars, with melody, chords, bass, and a drum part. The generator varies key, tempo, mode, progression, rhythm, register, and melodic contour. WAV is mono 16 kHz to keep the dataset manageable.

In [ ]:
MAJOR=[0,2,4,5,7,9,11]; MINOR=[0,2,3,5,7,8,10]
PROGS=[[0,4,5,3],[0,5,3,4],[5,3,0,4],[0,3,4,0],[1,4,0,5],[0,6,3,7]]
def make_piece(i):
    key=random.randrange(12); minor=random.random()<0.35; scale=MINOR if minor else MAJOR
    bpm=random.randint(78,150); beat=60/bpm; pm=pretty_midi.PrettyMIDI(initial_tempo=bpm)
    melody=pretty_midi.Instrument(program=random.choice([0,4,5,10,11,46]))
    chords=pretty_midi.Instrument(program=random.choice([0,4,19,48,89]))
    bass=pretty_midi.Instrument(program=random.choice([32,33,38]))
    drums=pretty_midi.Instrument(program=0,is_drum=True)
    prog=random.choice(PROGS); tonic=key
    for bar in range(8):
        degree=prog[bar%4]; root=tonic+scale[degree%7]
        third=scale[(degree+2)%7]; fifth=scale[(degree+4)%7]
        t=bar*4*beat
        for off in [0,1,2,3]:
            st=t+off*beat; dur=beat*0.88
            for p in [root+48, tonic+third+48, tonic+fifth+48]:
                chords.notes.append(pretty_midi.Note(velocity=random.randint(55,88),pitch=p,start=st,end=st+dur))
        bass.notes.append(pretty_midi.Note(velocity=92,pitch=root+24,start=t,end=t+beat*1.8))
        bass.notes.append(pretty_midi.Note(velocity=78,pitch=root+24,start=t+2*beat,end=t+2*beat+beat*0.8))
        for step in range(16):
            st=t+step*beat/4
            if step%4==0: drums.notes.append(pretty_midi.Note(velocity=90,pitch=36,start=st,end=st+0.08))
            if step%8==4: drums.notes.append(pretty_midi.Note(velocity=78,pitch=38,start=st,duration=0.08))
            if random.random()<0.72: drums.notes.append(pretty_midi.Note(velocity=random.randint(35,68),pitch=42,start=st,end=st+0.05))
        for step in range(8):
            if random.random()<0.86:
                deg=random.choice([0,1,2,3,4,5,6]); octave=random.choice([0,12]); pitch=key+scale[deg]+60+octave
                st=t+step*beat/2; dur=beat*random.choice([0.22,0.42,0.7])
                melody.notes.append(pretty_midi.Note(velocity=random.randint(62,112),pitch=pitch,start=st,end=st+dur))
    for ins in [melody,chords,bass,drums]: pm.instruments.append(ins)
    return pm, {'id':i,'bpm':bpm,'key':key,'mode':'minor' if minor else 'major','progression':'-'.join(map(str,prog))}
# Resume safely: completed MIDI/WAV pairs are retained across Colab reconnects.
manifest_path=ROOT/'manifest.csv'
if manifest_path.exists():
    old=pd.read_csv(manifest_path)
    rows=old.to_dict('records')
else:
    rows=[]
row_by_id={int(r['id']):r for r in rows}
for i in tqdm(range(2500)):
    # Always generate metadata in sequence so the seeded random stream remains reproducible.
    mid, meta=make_piece(i)
    mp=MIDI_DIR/f'{i:05d}.mid'; wp=WAV_DIR/f'{i:05d}.wav'
    if not mp.exists():
        mid.write(str(mp))
    if not wp.exists():
        render_path=f'/tmp/honeydew_render_{i:05d}.wav'
        cmd=['fluidsynth','-ni',SOUNDFONT,str(mp),'-F',render_path,'-r','16000']
        subprocess.run(cmd,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',render_path,'-ac','1','-ar','16000',str(wp)],check=True)
        if os.path.exists(render_path):
            os.remove(render_path)
    row_by_id[i]={**meta,'midi':f'midi/{mp.name}','wav':f'wav/{wp.name}'}
    if (i+1)%25==0 or i==2499:
        manifest=pd.DataFrame([row_by_id[k] for k in sorted(row_by_id)])
        manifest.to_csv(manifest_path,index=False)
manifest=pd.DataFrame([row_by_id[k] for k in sorted(row_by_id)])
manifest.to_csv(manifest_path,index=False)
print('MIDI:',len(list(MIDI_DIR.glob('*.mid'))),'WAV:',len(list(WAV_DIR.glob('*.wav'))))
print('Total size (GB):',round(sum(p.stat().st_size for p in ROOT.rglob('*') if p.is_file())/1e9,2))

## Train a small MIDI event language model
This baseline learns next-note pitch sequences from the generated MIDI files. It is not a finished music-composition model; evaluate its output before integrating it into Honeydew.

In [ ]:
from torch import nn
sequences=[]
for path in sorted(MIDI_DIR.glob('*.mid')):
    pm=pretty_midi.PrettyMIDI(str(path)); notes=[]
    for inst in pm.instruments:
        if not inst.is_drum:
            notes.extend(inst.notes)
    notes=sorted(notes,key=lambda n:n.start)
    seq=[min(127,max(0,n.pitch)) for n in notes]
    if len(seq)>8: sequences.append(seq[:256])
data=[torch.tensor(s,dtype=torch.long) for s in sequences]
class NoteLM(nn.Module):
    def __init__(self):
        super().__init__(); self.emb=nn.Embedding(128,96); self.rnn=nn.LSTM(96,192,batch_first=True); self.head=nn.Linear(192,128)
    def forward(self,x): return self.head(self.rnn(self.emb(x))[0])
model=NoteLM(); device='cuda' if torch.cuda.is_available() else 'cpu'; model.to(device)
opt=torch.optim.AdamW(model.parameters(),lr=2e-3); loss_fn=nn.CrossEntropyLoss()
# Split by source composition first to prevent windows from one song leaking across splits.
indices=list(range(len(data)))
random.Random(20261121).shuffle(indices)
cut=max(1,int(len(indices)*0.9))
train_ids=set(indices[:cut]); valid_ids=set(indices[cut:])
train=[]; valid=[]
for idx, seq in enumerate(data):
    target=train if idx in train_ids else valid
    for j in range(0,len(seq)-33,16):
        target.append(seq[j:j+33])
print('Training windows:',len(train),'Validation windows:',len(valid))
def epoch(items,training):
    model.train(training); losses=[]
    for start in range(0,len(items),128):
        batch=torch.stack(items[start:start+128]).to(device); x=batch[:,:-1]; y=batch[:,1:]
        with torch.set_grad_enabled(training):
            logits=model(x); loss=loss_fn(logits.reshape(-1,128),y.reshape(-1))
            if training: opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float('nan')
for e in range(5): print(f'epoch {e+1}: train={epoch(train,True):.3f} valid={epoch(valid,False):.3f}')
torch.save(model.cpu().state_dict(),ROOT/'honeydew_note_lm.pt')
print('Saved baseline model:',ROOT/'honeydew_note_lm.pt')

## Export
Create a ZIP for download or copy the folder to Google Drive. Keep the manifest with the audio/MIDI pairs. The notebook itself is stored in the Honeydew GitHub branch; generated binary data is created in your Colab runtime when you run all cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DEST=Path('/content/drive/MyDrive/Honeydew_2500_MIDI_2500_WAV')
if DEST.exists():
    shutil.rmtree(DEST)
shutil.copytree(ROOT,DEST)
print('Dataset copied to Google Drive:',DEST)
print('MIDI:',len(list((DEST/'midi').glob('*.mid'))))
print('WAV:',len(list((DEST/'wav').glob('*.wav'))))
print('Model:',(DEST/'honeydew_note_lm.pt').exists())
